In [1]:
!nvidia-smi

Fri Jul 17 18:29:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 5.0.0
huggingface_hub: 1.24.0


In [3]:
!pip uninstall -y torch torchvision torchaudio
!pip install --upgrade torch torchvision torchaudio

Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
  Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl (526.6 MB)
Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.7 MB)
Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.8 MB)


In [4]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

Torch: 2.13.0+cu130
TorchVision: 0.28.0+cu130


In [5]:
!pip install -q -U \
transformers \
datasets \
peft \
trl \
accelerate \
bitsandbytes \
sentencepiece

In [6]:
import torch
import transformers
import datasets
import peft
import huggingface_hub

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("HF Hub:", huggingface_hub.__version__)

Torch: 2.13.0+cu130
Transformers: 5.14.1
Datasets: 5.0.0
PEFT: 0.19.1
HF Hub: 1.24.0


In [7]:
import torch
import transformers
import datasets
import huggingface_hub
import peft

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)
print(huggingface_hub.__version__)
print(peft.__version__)

2.13.0+cu130
5.14.1
5.0.0
1.24.0
0.19.1


In [12]:
import transformers
from transformers import TrainingArguments
import inspect

print("Transformers:", transformers.__version__)
print(inspect.signature(TrainingArguments.__init__))

Transformers: 5.14.1
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: b

In [14]:
from huggingface_hub import login
login()

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

# ----------------------------------------------------
# Model
# ----------------------------------------------------

MODEL_NAME = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

# Llama has no pad token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

model.config.pad_token_id = tokenizer.pad_token_id

# ----------------------------------------------------
# LoRA
# ----------------------------------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------

dataset = load_dataset(
    "stanfordnlp/imdb",
    split="train[:1%]"
)


def preprocess(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

    outputs["labels"] = outputs["input_ids"].copy()

    return outputs


dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names,
)

# ----------------------------------------------------
# Data Collator
# ----------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# ----------------------------------------------------
# Training
# ----------------------------------------------------

training_args = TrainingArguments(
    output_dir="./lora-output",

    num_train_epochs=1,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    remove_unused_columns=False,

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# ----------------------------------------------------
# Train
# ----------------------------------------------------

trainer.train()

# ----------------------------------------------------
# Save
# ----------------------------------------------------

model.save_pretrained("lora-model")
tokenizer.save_pretrained("lora-model")

print("Training Complete!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 8,388,608 || all params: 6,746,804,224 || trainable%: 0.1243


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 94.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)